# Lab 06-10 · The Ideal Op-Amp

Companion to chapter [06-10](../../chapters/06-10-ideal-op-amp.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import numpy as np
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — How ideal is "ideal"?

For each configuration, the exact closed-loop gain with open-loop gain $A$ is the ideal gain divided by $1 + (\text{noise gain})/A$ (06-06, Problem 06-10.11).

**🔮 Predict:** for the inverting amplifier of Example 06-10.1 (gain −4.7, noise gain 5.7), how far from −4.7 is the gain at $A = 10^3$, $10^5$ and $10^8$?

In [2]:
def inverting_gain(R1, R2, A):
    return -(R2 / R1) / (1 + (1 + R2 / R1) / A)

def noninverting_gain(R1, R2, A):
    return (1 + R2 / R1) / (1 + (1 + R2 / R1) / A)

for A in (1e3, 1e5, 1e8):
    gi = inverting_gain(10, 47, A); gn = noninverting_gain(1, 100, A)
    print(f"A = {A:7.0e}: inverting {gi:.6f} (error {100 * (1 - gi / -4.7):.4f}%), "
          f"non-inverting 101 -> {gn:.4f} (error {100 * (1 - gn / 101):.4f}%)")
assert abs(inverting_gain(10, 47, 1e3) + 4.67336) < 1e-5

if HAVE_NGSPICE:
    out = subprocess.run(["ngspice", "-b", "opamps.cir"], capture_output=True, text=True).stdout
    ng = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) == 2:
            try:
                ng[parts[0].strip()] = float(parts[1])
            except ValueError:
                pass
    print("ngspice (A = 1e8): Example 06-10.1 output", ng["ex1out"], "V; Problem 06-10.11 (A = 1e4) gain", ng["p11gain"])
else:
    print("ngspice not found; see SETUP.md. Skipping the simulator check.")

A =   1e+03: inverting -4.673362 (error 0.5668%), non-inverting 101 -> 91.7348 (error 9.1735%)
A =   1e+05: inverting -4.699732 (error 0.0057%), non-inverting 101 -> 100.8981 (error 0.1009%)
A =   1e+08: inverting -4.700000 (error 0.0000%), non-inverting 101 -> 100.9999 (error 0.0001%)
ngspice (A = 1e8): Example 06-10.1 output -2.35 V; Problem 06-10.11 (A = 1e4) gain -99.0001


At $A = 10^3$ the inverting stage is 0.57% low; by $10^5$ the error is 57 ppm and by $10^8$ negligible. The relevant ratio is always noise gain over $A$.

## Part 2 — When the virtual short fails

Sweep the input of Example 06-10.1 with an output that saturates smoothly at ±12 V ($v_{out} = 12\tanh(A(v_+ - v_-)/12)$, $A = 10^5$).

**🔮 Predict:** at what input does $v_-$ leave 0 V, and how does it behave beyond?

In [3]:
R1, R2, A, VS = 10.0, 47.0, 1e5, 12.0
def solve_inverting(vin):
    # KCL at the - input: (vin - vm)/R1 + (vout - vm)/R2 = 0 with vout = VS*tanh(A*(0 - vm)/VS)
    f = lambda vm: (vin - vm) / R1 + (VS * np.tanh(A * (0 - vm) / VS) - vm) / R2
    vm = brentq(f, -5, 5, xtol=1e-15)
    return vm, VS * np.tanh(-A * vm / VS)

vins = np.linspace(-4, 4, 401)
res = np.array([solve_inverting(x) for x in vins])
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2))
axes[0].plot(vins, res[:, 1]); axes[0].set_xlabel("input (V)"); axes[0].set_ylabel("output (V)"); axes[0].grid(True, alpha=0.3)
axes[1].plot(vins, res[:, 0] * 1e3); axes[1].set_xlabel("input (V)"); axes[1].set_ylabel("v− (mV)"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
vm3, vo3 = solve_inverting(3.0)
print(f"at 3 V in: output {vo3:.3f} V, v- = {vm3:.4f} V (virtual short lost); linear limit 12/4.7 = {12 / 4.7:.3f} V")
assert abs(vm3 - (3 * 47 - 12 * 10) / 57) < 1e-3

at 3 V in: output -12.000 V, v- = 0.3684 V (virtual short lost); linear limit 12/4.7 = 2.553 V


Up to about ±2.55 V the − input stays within microvolts of 0 V. Beyond, the output is pinned at a rail, and $v_-$ simply follows the resistive divider between the input and the stuck output: 0.368 V at 3 V in. Any analysis that assumes the virtual short is then wrong.

## Part 3 — CMRR with real resistors

**🔮 Predict:** with independent ±1% (uniform) resistors, how does the typical CMRR of Example 06-10.3's difference amplifier compare with the 67.9 dB worst case?

In [4]:
rng = np.random.default_rng(610)
N = 20000
R = np.array([1.0, 100.0, 1.0, 100.0]) * (1 + rng.uniform(-0.01, 0.01, (N, 4)))
a = R[:, 1] / R[:, 0]; b = R[:, 3] / R[:, 2]
acm = (b - a) / (1 + b)                       # common-mode gain, derived in the chapter
cmrr_db = 20 * np.log10(100 / np.abs(acm))
print(f"median CMRR {np.median(cmrr_db):.1f} dB; 1st percentile {np.percentile(cmrr_db, 1):.1f} dB; "
      f"worst case 67.9 dB")
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(np.clip(cmrr_db, 60, 120), bins=60)
ax.axvline(67.9, color="0.3", ls="--", lw=0.8); ax.set_xlabel("CMRR (dB)"); ax.set_ylabel("count")
plt.show()
assert np.percentile(cmrr_db, 1) > 67.8

median CMRR 82.1 dB; 1st percentile 71.0 dB; worst case 67.9 dB


The median unit (82 dB) is about 14 dB better than the worst case, and even the 1st percentile (71 dB) clears it, because four independent errors rarely line up. Statistical design (06-07) would quote the distribution; a guaranteed specification must use the worst case, or matched resistor networks.

## Part 4 — The error budget with signs

Example 06-10.4 added the LM358's three error terms by magnitude, 984.5 mV. The bias currents of a PNP-input op-amp both flow *out* of the inputs, so their signs are known.

**🔮 Predict:** with both bias currents flowing out of the inputs, do the two bias-current terms add or partly cancel?

In [5]:
vos, ib = 7e-3, 250e-9                 # LM358 maxima at 25 C; current out of each input
G, R2, Rs = 101.0, 100e3, 10e3
term_vos = vos * G
term_ibm = -ib * R2                    # current out of the - input: the output must pull it through R2
term_ibp = ib * Rs * G                 # current out of the + input raises v+ by ib*Rs
print(f"offset {term_vos * 1e3:.1f} mV, - bias {term_ibm * 1e3:.1f} mV, + bias {term_ibp * 1e3:.1f} mV")
print(f"signed total {1e3 * (term_vos + term_ibm + term_ibp):.1f} mV versus sum of magnitudes {1e3 * (abs(term_vos) + abs(term_ibm) + abs(term_ibp)):.1f} mV")
assert abs(1e3 * (term_vos + term_ibm + term_ibp) - 934.5) < 1e-9

offset 707.0 mV, - bias -25.0 mV, + bias 252.5 mV
signed total 934.5 mV versus sum of magnitudes 984.5 mV


The two bias terms have opposite signs and partly cancel (934.5 mV instead of 984.5 mV). The offset voltage's sign varies from unit to unit, so it still enters as a magnitude. Adding magnitudes is the safe worst case when signs are unknown; when physics fixes them, a signed budget is tighter. Choosing the source resistances equal ($R_s = R_1\parallel R_2$, Problem 06-10.8) would cancel the bias terms almost entirely.

## Your turn

1. In Part 1, what open-loop gain keeps a gain-of-1000 non-inverting amplifier within 0.1%?
2. In Part 3, repeat with 0.1% resistors. By how much does the worst case improve?
3. In Part 4, change $R_s$ to $R_1\parallel R_2$ = 990 Ω. What are the bias terms now?

<details><summary>Answers</summary>

1. The error is about $1000/A$, so $A \ge 10^6$ (120 dB).
2. Tenfold, 20 dB: the worst-case CMRR becomes about 88 dB, since $(1 + G)/(4t)$ scales as $1/t$.
3. The + term becomes $250\ \text{nA}\times990\ \Omega\times101 = 25.0$ mV, cancelling the −25 mV − term: the residual is set by the offset current $I_{OS}$ (not modeled here).

</details>